# Report dataset sizes

This notebook reports file sizes for the external beach-cell dataset without reading matrix contents. It uses filesystem metadata only. It prints the total external dataset size, per-date folder sizes, and the size of files corresponding to annotated cells from `Data/Raw/Descriptor.csv`.

In [1]:
from pathlib import Path
import os
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Data").exists():
    raise RuntimeError("Could not locate project root containing Data directory.")

CELL_DATA_ROOT = Path("/Volumes/BACKUPS/Piotr_beach/beach_all")
DESCRIPTOR_PATH = PROJECT_ROOT / "Data" / "Raw" / "Descriptor.csv"
FEATURE_SUFFIXES = ("r", "g", "b", "s", "d")

print(f"Project root:      {PROJECT_ROOT}")
print(f"External dataset:  {CELL_DATA_ROOT}")
print(f"Descriptor:        {DESCRIPTOR_PATH}")

Project root:      /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software
External dataset:  /Volumes/BACKUPS/Piotr_beach/beach_all
Descriptor:        /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Data/Raw/Descriptor.csv


In [2]:
def format_size(num_bytes):
    """Return both decimal TB/GB and binary TiB/GiB for clarity."""
    num_bytes = int(num_bytes)
    return (
        f"{num_bytes / 1_000_000_000_000:.4f} TB "
        f"({num_bytes / 1024**4:.4f} TiB, "
        f"{num_bytes / 1_000_000_000:.2f} GB)"
    )

def iter_files(root):
    """Yield files recursively using os.scandir for lower overhead than pathlib.rglob."""
    stack = [Path(root)]
    while stack:
        current = stack.pop()
        try:
            with os.scandir(current) as entries:
                for entry in entries:
                    try:
                        if entry.is_dir(follow_symlinks=False):
                            stack.append(Path(entry.path))
                        elif entry.is_file(follow_symlinks=False):
                            yield Path(entry.path), entry.stat(follow_symlinks=False).st_size
                    except OSError as exc:
                        print(f"Skipping unreadable entry {entry.path}: {exc}")
        except OSError as exc:
            print(f"Skipping unreadable directory {current}: {exc}")

def directory_size(root):
    total = 0
    file_count = 0
    for _, size in iter_files(root):
        total += size
        file_count += 1
    return total, file_count

In [3]:
if not CELL_DATA_ROOT.exists():
    raise FileNotFoundError(f"External dataset root not found: {CELL_DATA_ROOT}")

date_dirs = sorted([path for path in CELL_DATA_ROOT.iterdir() if path.is_dir()], key=lambda p: p.name)
print(f"Date folders found: {len(date_dirs):,}")

rows = []
for date_dir in date_dirs:
    size_bytes, file_count = directory_size(date_dir)
    rows.append(
        {
            "date": date_dir.name,
            "file_count": file_count,
            "size_bytes": size_bytes,
            "size_gb": size_bytes / 1_000_000_000,
            "size_tib": size_bytes / 1024**4,
        }
    )
    print(f"{date_dir.name}: {file_count:,} files, {format_size(size_bytes)}")

date_size_df = pd.DataFrame(rows)
total_size = int(date_size_df["size_bytes"].sum()) if not date_size_df.empty else 0
total_files = int(date_size_df["file_count"].sum()) if not date_size_df.empty else 0

print("\nTotal external dataset size")
print(f"Files: {total_files:,}")
print(f"Size:  {format_size(total_size)}")

date_size_df

Date folders found: 15
20180722: 232,415 files, 0.0008 TB (0.0008 TiB, 0.85 GB)
20180731: 271,520 files, 0.0010 TB (0.0009 TiB, 0.99 GB)
20180812: 335,110 files, 0.0012 TB (0.0011 TiB, 1.22 GB)
20180826: 314,075 files, 0.0011 TB (0.0010 TiB, 1.15 GB)
20180910: 420,980 files, 0.0015 TB (0.0014 TiB, 1.52 GB)
20180915: 262,160 files, 0.0010 TB (0.0009 TiB, 0.96 GB)
20190822: 306,785 files, 0.0011 TB (0.0010 TiB, 1.12 GB)
20220826: 374,875 files, 0.0014 TB (0.0013 TiB, 1.40 GB)
20220910: 250,350 files, 0.0009 TB (0.0008 TiB, 0.93 GB)
20220924: 253,825 files, 0.0009 TB (0.0008 TiB, 0.93 GB)
20230613: 108,050 files, 0.0004 TB (0.0004 TiB, 0.39 GB)
20230731: 199,720 files, 0.0007 TB (0.0007 TiB, 0.74 GB)
20230807: 47,530 files, 0.0002 TB (0.0002 TiB, 0.18 GB)
20240623: 360,215 files, 0.0013 TB (0.0012 TiB, 1.33 GB)
20240806: 290,745 files, 0.0011 TB (0.0010 TiB, 1.07 GB)

Total external dataset size
Files: 4,028,355
Size:  0.0148 TB (0.0135 TiB, 14.80 GB)


,date,file_count,size_bytes,size_gb,size_tib
0,20180722,232415,846507254,0.846507,0.000770
1,20180731,271520,994813900,0.994814,0.000905
2,20180812,335110,1224057778,1.224058,0.001113
3,20180826,314075,1149896235,1.149896,0.001046
4,20180910,420980,1524851663,1.524852,0.001387
5,20180915,262160,960461106,0.960461,0.000874
6,20190822,306785,1123221648,1.123222,0.001022
7,20220826,374875,1397371818,1.397372,0.001271
8,20220910,250350,932541441,0.932541,0.000848
9,20220924,253825,927934450,0.927934,0.000844


## Annotated Data Size

The annotated size is calculated from `Descriptor.csv`. For every annotated `(date, cell_id)` pair, the notebook checks the five expected files: `r`, `g`, `b`, `s`, and `d`. Missing files are reported but not copied.

In [5]:
if not DESCRIPTOR_PATH.exists():
    raise FileNotFoundError(f"Descriptor file not found: {DESCRIPTOR_PATH}")

# Descriptor.csv in this project is semicolon-separated and has no header:
# date;cell_id;class
try:
    descriptor = pd.read_csv(DESCRIPTOR_PATH, dtype=str, sep=None, engine="python", header=None, encoding="utf-8-sig")
except UnicodeDecodeError:
    descriptor = pd.read_csv(DESCRIPTOR_PATH, dtype=str, sep=None, engine="python", header=None)

if descriptor.shape[1] < 2:
    raise ValueError(
        f"Descriptor.csv must contain at least date and cell_id columns; got shape {descriptor.shape}."
    )

descriptor = descriptor.iloc[:, :3].copy()
descriptor.columns = ["date", "cell_id", "class"][: descriptor.shape[1]]

annotated = descriptor[["date", "cell_id"]].dropna().copy()
annotated["date"] = annotated["date"].astype(str).str.replace("\ufeff", "", regex=False).str.strip()
annotated["cell_id"] = annotated["cell_id"].astype(str).str.strip()
annotated = annotated.drop_duplicates()

print(f"Annotated cells in descriptor: {len(annotated):,}")
print(f"Dates with annotated cells:    {annotated['date'].nunique():,}")
display(annotated.head())

Annotated cells in descriptor: 4,938
Dates with annotated cells:    15


,date,cell_id
0,20180722,319007
1,20180722,320534
2,20180722,343426
3,20180722,346483
4,20180722,471840


In [6]:
annotated_rows = []
missing_rows = []

for row in annotated.itertuples(index=False):
    cell_total = 0
    files_found = 0
    for suffix in FEATURE_SUFFIXES:
        file_path = CELL_DATA_ROOT / row.date / f"{row.cell_id}{suffix}.txt"
        try:
            stat = file_path.stat()
        except FileNotFoundError:
            missing_rows.append({"date": row.date, "cell_id": row.cell_id, "feature": suffix, "path": str(file_path)})
            continue
        except OSError as exc:
            missing_rows.append({"date": row.date, "cell_id": row.cell_id, "feature": suffix, "path": str(file_path), "error": str(exc)})
            continue
        cell_total += stat.st_size
        files_found += 1

    annotated_rows.append(
        {
            "date": row.date,
            "cell_id": row.cell_id,
            "files_found": files_found,
            "size_bytes": cell_total,
        }
    )

annotated_size_df = pd.DataFrame(annotated_rows)
missing_annotated_files_df = pd.DataFrame(missing_rows)

annotated_by_date_df = (
    annotated_size_df
    .groupby("date", as_index=False)
    .agg(
        annotated_cells=("cell_id", "count"),
        files_found=("files_found", "sum"),
        size_bytes=("size_bytes", "sum"),
    )
)
annotated_by_date_df["size_gb"] = annotated_by_date_df["size_bytes"] / 1_000_000_000
annotated_by_date_df["size_tib"] = annotated_by_date_df["size_bytes"] / 1024**4

annotated_total_size = int(annotated_by_date_df["size_bytes"].sum()) if not annotated_by_date_df.empty else 0
annotated_total_files = int(annotated_by_date_df["files_found"].sum()) if not annotated_by_date_df.empty else 0

print("Annotated external data size")
print(f"Files found: {annotated_total_files:,}")
print(f"Size:        {format_size(annotated_total_size)}")
print(f"Missing expected annotated files: {len(missing_annotated_files_df):,}")

display(annotated_by_date_df)
if not missing_annotated_files_df.empty:
    display(missing_annotated_files_df.head(50))

Annotated external data size
Files found: 3,176
Size:        0.0000 TB (0.0000 TiB, 0.01 GB)
Missing expected annotated files: 21,514


,date,annotated_cells,files_found,size_bytes,size_gb,size_tib
0,20180722,311,1555,5686900,0.005687,0.000005
1,20180731,359,1621,5878719,0.005879,0.000005
2,20180812,355,0,0,0.000000,0.000000
3,20180826,380,0,0,0.000000,0.000000
4,20180910,270,0,0,0.000000,0.000000
5,20180915,311,0,0,0.000000,0.000000
6,20190822,385,0,0,0.000000,0.000000
7,20220826,357,0,0,0.000000,0.000000
8,20220910,260,0,0,0.000000,0.000000
9,20220924,320,0,0,0.000000,0.000000


,date,cell_id,feature,path
0,20180731,1179843,g,/Volumes/BACKUPS/Piotr_beach/beach_all/2018073...
1,20180731,1179843,b,/Volumes/BACKUPS/Piotr_beach/beach_all/2018073...
2,20180731,1179843,s,/Volumes/BACKUPS/Piotr_beach/beach_all/2018073...
3,20180731,1179843,d,/Volumes/BACKUPS/Piotr_beach/beach_all/2018073...
4,20180731,1813141,r,/Volumes/BACKUPS/Piotr_beach/beach_all/2018073...
5,20180731,1813141,g,/Volumes/BACKUPS/Piotr_beach/beach_all/2018073...
6,20180731,1813141,b,/Volumes/BACKUPS/Piotr_beach/beach_all/2018073...
7,20180731,1813141,s,/Volumes/BACKUPS/Piotr_beach/beach_all/2018073...
8,20180731,1813141,d,/Volumes/BACKUPS/Piotr_beach/beach_all/2018073...
9,20180731,1814672,r,/Volumes/BACKUPS/Piotr_beach/beach_all/2018073...
